In [9]:
# import lib
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, f1_score,
    recall_score, precision_score
)

import warnings
warnings.filterwarnings("ignore")

In [10]:
def load_and_explore_dataset(filepath):
    print("=" * 60)
    print("LOAD AND  EXPLORE DATASET")
    print("=" * 60)

    df = pd.read_csv(filepath)

    print("Shape of the dataset")
    print(df.shape)
    print("\nCheck for missing values")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive stats")
    print(df.describe())
    print("\nDataset Info")
    print(df.info())
    print("\nStatus Distribution:")
    print(df["status"].value_counts())
    print("\nStatus percentage Distribution:")
    print(df["status"].value_counts(normalize=True) * 100)

    return df

In [11]:
def identify_features(df):
    print("=" * 60)
    print("IDENTIFY FEATURE DATASET")
    print("=" * 60)

    numerical_features = [
        "total_income",
        "applicant_age",
        "years_of_working",
        "total_bad_debt",
        "total_good_debt"
    ]

    categorical_features = [
        "applicant_gender",
        "education_type",
        "family_status",
        "income_type",
        "job_title"
    ]

    target_column = "status"

    X = df[numerical_features + categorical_features]
    y = df[target_column]

    feature_columns = numerical_features + categorical_features
    
    print("Features Shape:", X.shape)
    print("Target Shape:", y.shape)

    return X, y, numerical_features, categorical_features, feature_columns

In [12]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size", X_train.shape[0])
    print("Testing set size", X_test.shape[0])
    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.min()), float(y_train.max())
    ))
    print("\nTesting set price range {:.2f} - {:.2f}".format(
        float(y_test.min()), float(y_test.max())
    ))

    return X_train, X_test, y_train, y_test

In [13]:
def build_and_train_model(X_train, y_train, numerical_features, categorical_features, feature_columns):
    print("\n" + "=" * 60)
    print("BUILDING & TRAINING PIPELINE")
    print("=" * 60)

    numerical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    
    model = Pipeline(steps=[
       ("preprocessor", preprocessor),
       ("classifier", XGBClassifier(
           random_state=42,
           use_label_encoder=False,
           eval_metric="aucpr",
           scale_pos_weight=99.593/0.406,
           n_estimators=500,
           max_depth=8,
           max_delta_step=4
       ))
    ])
    model.fit(X_train, y_train)
    print("Model trained successfully")
    
    print("\nxg_model coefficients")
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    importances = model.named_steps["classifier"].feature_importances_

    xg_importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    xg_importance = xg_importance.sort_values(
        by="Importance",
        ascending=False
    )
    
    print(xg_importance.head(10))
        
    return model

In [14]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    print("\n" + "=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    y_train_pred  = model.predict(X_train)
    y_test_pred   = model.predict(X_test)
    
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba  = model.predict_proba(X_test)[:, 1]

    metrics = {
        "pr_auc":  (average_precision_score(y_train, y_train_proba),
                    average_precision_score(y_test,  y_test_proba)),
        "recall":  (recall_score(y_train, y_train_pred),
                    recall_score(y_test,  y_test_pred)),
        "f1":      (f1_score(y_train, y_train_pred),
                    f1_score(y_test,  y_test_pred)),
    }

    print(f"\n{'Metric':<12} {'Train':>10} {'Test':>10}")
    print("-" * 34)
    for name, (train_val, test_val) in metrics.items():
        print(f"{name.upper():<12} {train_val:>10.4f} {test_val:>10.4f}")

    # test classification report
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT — TEST")
    print("=" * 60)
    print(classification_report(y_test, y_test_pred, target_names=["Negative", "Positive"]))

    # confusion metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
    print(f"  TP: {tp}  |  FP: {fp}")
    print(f"  FN: {fn}  |  TN: {tn}")

    # Stratified CV on PR-AUC only
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_pr_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"\n  CV PR-AUC:  {cv_pr_auc.mean():.4f} ± {cv_pr_auc.std():.4f}")

    return {
        "pr_auc_train":  metrics["pr_auc"][0],
        "pr_auc_test":   metrics["pr_auc"][1],
        "recall_train":  metrics["recall"][0],
        "recall_test":   metrics["recall"][1],
        "f1_train":      metrics["f1"][0],
        "f1_test":       metrics["f1"][1],
        "cv_pr_auc":     cv_pr_auc,
        "y_test_pred":   y_test_pred,
        "y_test_proba":  y_test_proba,
    }

In [15]:
def save_model_artifact(model):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "../models/xg_model_pipeline.pkl")
    print("Pipeline model saved successfully")

    print("\n" + "=" * 60)
    print("ALL MODEL ARTIFACT SAVED SUCCESSFULLY")
    print("=" * 60)

In [16]:
def main():
    filepath = "../data/cleaned/cleaned_credit_card_approval.csv"

    df = load_and_explore_dataset(filepath)

    X, y, numerical_features, categorical_features, feature_columns = identify_features(df)

    X_train, X_test, y_train, y_test = split_data(X, y)

    model = build_and_train_model(X_train, y_train, numerical_features, categorical_features, feature_columns)

    evaluate_model(model, X_train, X_test, y_train, y_test)

    save_model_artifact(model)


if __name__ == "__main__":
    main()

LOAD AND  EXPLORE DATASET
Shape of the dataset
(17452, 11)

Check for missing values
applicant_gender    6970
total_income        6967
income_type         6974
education_type      6954
family_status       6974
job_title           6996
applicant_age       6977
years_of_working    6960
total_bad_debt      6987
total_good_debt     7070
status                 0
dtype: int64

First five rows:
  applicant_gender  total_income    income_type  \
0           Female           NaN            NaN   
1             Male      180000.0        Working   
2              NaN      112500.0            NaN   
3           Female      270000.0  State Servant   
4              NaN           NaN        Working   

                  education_type family_status job_title  applicant_age  \
0                            NaN       Married       NaN           53.0   
1  Secondary / Secondary Special       Married  Managers           45.0   
2                            NaN           NaN       NaN            NaN   
3 